# Transforms & Preprocessing Notebook

This notebook contains feature engineering and preprocessing steps.

In [86]:
import numpy as np
print(np.__version__)
import pandas as pd
from sklearn.preprocessing import StandardScaler

2.0.1


In [87]:
# Load the original dataset
df = pd.read_csv('../../data/online_shoppers_intention.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()


Dataset shape: (12330, 18)

Columns: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']

First few rows:


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [88]:
# normalize bounce rates using standard scaler
scaler = StandardScaler()
df['BounceRates'] = scaler.fit_transform(df[['BounceRates']])

In [89]:
count_cols = ["Administrative", "Informational", "ProductRelated"]
duration_cols = ["Administrative_Duration", "Informational_Duration", "ProductRelated_Duration"]

def closure(df, cols):
    subset = df[cols]
    sums = subset.sum(axis=1)

    closed = subset.div(sums.replace(0, pd.NA), axis=0)
    return closed

# Apply closure
df_counts_closed = closure(df, count_cols).add_suffix("_closed")
df_durs_closed   = closure(df, duration_cols).add_suffix("_closed")

# Append back to main dataframe
df_closure = pd.concat([df, df_counts_closed, df_durs_closed], axis=1)

df_closure.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,TrafficType,VisitorType,Weekend,Revenue,Administrative_closed,Informational_closed,ProductRelated_closed,Administrative_Duration_closed,Informational_Duration_closed,ProductRelated_Duration_closed
0,0,0.0,0,0.0,1,0.000000,3.667189,0.20,0.0,0.0,...,1,Returning_Visitor,False,False,0.0,0.0,1.0,<NA>,<NA>,<NA>
1,0,0.0,0,0.0,2,64.000000,-0.457683,0.10,0.0,0.0,...,2,Returning_Visitor,False,False,0.0,0.0,1.0,0.0,0.0,1.0
2,0,0.0,0,0.0,1,0.000000,3.667189,0.20,0.0,0.0,...,3,Returning_Visitor,False,False,0.0,0.0,1.0,<NA>,<NA>,<NA>
3,0,0.0,0,0.0,2,2.666667,0.573535,0.14,0.0,0.0,...,4,Returning_Visitor,False,False,0.0,0.0,1.0,0.0,0.0,1.0
4,0,0.0,0,0.0,10,627.500000,-0.045196,0.05,0.0,0.0,...,4,Returning_Visitor,True,False,0.0,0.0,1.0,0.0,0.0,1.0


In [90]:
df_closure.isna().sum()

Administrative                      0
Administrative_Duration             0
Informational                       0
Informational_Duration              0
ProductRelated                      0
ProductRelated_Duration             0
BounceRates                         0
ExitRates                           0
PageValues                          0
SpecialDay                          0
Month                               0
OperatingSystems                    0
Browser                             0
Region                              0
TrafficType                         0
VisitorType                         0
Weekend                             0
Revenue                             0
Administrative_closed               6
Informational_closed                6
ProductRelated_closed               6
Administrative_Duration_closed    720
Informational_Duration_closed     720
ProductRelated_Duration_closed    720
dtype: int64

In [91]:
# take the rows which have NaN in their *_closed columns and put them into a seprate df
df_closure_na = df_closure[df_closure.isna().any(axis=1)]
df_closure_na.head()

# drop the rows with NaN in their *_closed columns
df_closure = df_closure.dropna(subset=df_closure.columns[df_closure.columns.str.endswith("_closed")])

df_closure.shape

(11610, 24)

In [92]:


def multiplicative_replace_closed(row, cols, delta=1e-6):
    x = row[cols].to_numpy(dtype=float)
    D = len(x)

    zeros = (x == 0.0)
    z = zeros.sum()

    # If no zeros, just re-close for numerical exactness
    if z == 0:
        s = x.sum()
        return x / s if s > 0 else np.ones(D) / D

    # Save original nonzero mass BEFORE replacement
    nonzero_idx = ~zeros
    nz_mass_orig = x[nonzero_idx].sum()

    # If all zeros (shouldn't happen after proper closure), return uniform
    if nz_mass_orig <= 0:
        return np.ones(D) / D

    mass_added = delta * z

    # Guard: cannot take more mass than exists
    if mass_added >= nz_mass_orig:
        # fallback: uniform (or choose smaller delta)
        return np.ones(D) / D

    # Replace zeros with delta
    x[zeros] = delta

    # Shrink original nonzeros proportionally to free up mass_added
    shrink = 1 - (mass_added / nz_mass_orig)
    x[nonzero_idx] = x[nonzero_idx] * shrink

    # Final closure
    x = x / x.sum()
    return x


In [93]:
# Apply ALR transform on closed counts and durations
# ALR with ProductRelated as reference: η₁ = log(s_Adm/s_Prod), η₂ = log(s_Info/s_Prod)

# Ensure numpy is properly imported (guard against shadowing)
import numpy as np
assert hasattr(np, 'log'), "numpy module is not properly imported or has been shadowed"

def alr_transform(df, closed_cols, reference_idx=2):
    """
    Apply additive log-ratio (ALR) transform to closed compositional data.
    
    Parameters:
    df: DataFrame with closed compositional columns
    closed_cols: list of closed column names (e.g., ['Administrative_closed', 'Informational_closed', 'ProductRelated_closed'])
    reference_idx: index of reference part (default 2 = ProductRelated)
    
    Returns:
    DataFrame with ALR transformed columns only
    """
    # Diagnostics: check numpy import
    print(f"Diagnostics - type(np): {type(np)}, np.log exists: {hasattr(np, 'log')}")
    
    # Force numeric dtype and create a copy
    closed_df = df[closed_cols].astype('float64').copy()
    
    # Diagnostics: check dtypes
    print(f"Diagnostics - closed_df dtypes:\n{closed_df.dtypes}")
    
    # Guard: ensure all columns are numeric
    non_numeric = closed_df.select_dtypes(exclude=[np.number]).columns
    if len(non_numeric) > 0:
        raise TypeError(f"Non-numeric columns found: {list(non_numeric)}. Cannot apply ALR transform.")
    
    reference_col = closed_cols[reference_idx]
    
    # Detect rows with zeros ANYWHERE in the compositional vector (not just reference)
    rows_with_zeros = (closed_df[closed_cols] == 0).any(axis=1)
    
    # Apply multiplicative replacement ONLY to rows that contain zeros
    if rows_with_zeros.any():
        for idx in closed_df[rows_with_zeros].index:
            row = closed_df.loc[idx]
            replaced = multiplicative_replace_closed(row, closed_cols)
            closed_df.loc[idx] = replaced
            
            # Verify positivity and closure for replaced rows
            assert (replaced > 0).all(), f"Row {idx}: replacement resulted in non-positive values"
            assert np.isclose(replaced.sum(), 1.0, rtol=1e-10), f"Row {idx}: replacement not closed (sum={replaced.sum()})"
    
    # Guard: ensure all values are positive before taking log
    if (closed_df[closed_cols] <= 0).any().any():
        raise ValueError("Non-positive values detected in closed compositional data. Cannot apply log transform.")
    
    # Compute ALR transform: log(s_i / s_ref) for all i != ref
    alr_cols = []
    ref_values = closed_df[reference_col].to_numpy(dtype=float)
    
    for i, col in enumerate(closed_cols):
        if i != reference_idx:
            alr_col = col.replace('_closed', '_alr')
            col_values = closed_df[col].to_numpy(dtype=float)
            ratio = col_values / ref_values
            
            # Diagnostics before log
            print(f"Diagnostics - ratio dtype: {ratio.dtype}, ratio[:5]: {ratio[:5]}")
            print(f"Diagnostics - type(np.log): {type(np.log)}")
            
            # Compute log transform using numpy on float arrays
            alr_values = np.log(ratio)
            closed_df[alr_col] = alr_values
            alr_cols.append(alr_col)
    
    # Return ONLY the ALR columns
    result = closed_df[alr_cols].copy()
    
    # Safety check: ensure all ALR values are finite
    if not np.isfinite(result.values).all():
        invalid_mask = ~np.isfinite(result.values)
        invalid_rows = np.where(invalid_mask.any(axis=1))[0]
        raise ValueError(f"ALR transform produced non-finite values (inf/-inf/NaN) in rows: {invalid_rows}")
    
    return result

# Apply ALR to counts
count_closed_cols = [f"{col}_closed" for col in count_cols]
df_counts_alr = alr_transform(df_closure, count_closed_cols, reference_idx=2)

# Apply ALR to durations  
duration_closed_cols = [f"{col}_closed" for col in duration_cols]
df_durs_alr = alr_transform(df_closure, duration_closed_cols, reference_idx=2)

# Append ALR transformed features to dataframe
df_alr = pd.concat([df_closure, df_counts_alr, df_durs_alr], axis=1)

print("ALR transformed columns:")
print(list(df_counts_alr.columns) + list(df_durs_alr.columns))
df_alr.head()


Diagnostics - type(np): <class 'module'>, np.log exists: True
Diagnostics - closed_df dtypes:
Administrative_closed    float64
Informational_closed     float64
ProductRelated_closed    float64
dtype: object
Diagnostics - ratio dtype: float64, ratio[:5]: [1.000002e-06 1.000002e-06 1.000002e-06 1.000002e-06 1.000002e-06]
Diagnostics - type(np.log): <class 'numpy.ufunc'>
Diagnostics - ratio dtype: float64, ratio[:5]: [1.000002e-06 1.000002e-06 1.000002e-06 1.000002e-06 1.000002e-06]
Diagnostics - type(np.log): <class 'numpy.ufunc'>
Diagnostics - type(np): <class 'module'>, np.log exists: True
Diagnostics - closed_df dtypes:
Administrative_Duration_closed    float64
Informational_Duration_closed     float64
ProductRelated_Duration_closed    float64
dtype: object
Diagnostics - ratio dtype: float64, ratio[:5]: [1.000002e-06 1.000002e-06 1.000002e-06 1.000002e-06 1.000002e-06]
Diagnostics - type(np.log): <class 'numpy.ufunc'>
Diagnostics - ratio dtype: float64, ratio[:5]: [1.000002e-06 1.0000

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,Administrative_closed,Informational_closed,ProductRelated_closed,Administrative_Duration_closed,Informational_Duration_closed,ProductRelated_Duration_closed,Administrative_alr,Informational_alr,Administrative_Duration_alr,Informational_Duration_alr
1,0,0.0,0,0.0,2,64.000000,-0.457683,0.100000,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,-13.815509,-13.815509,-13.815509,-13.815509
3,0,0.0,0,0.0,2,2.666667,0.573535,0.140000,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,-13.815509,-13.815509,-13.815509,-13.815509
4,0,0.0,0,0.0,10,627.500000,-0.045196,0.050000,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,-13.815509,-13.815509,-13.815509,-13.815509
5,0,0.0,0,0.0,19,154.216667,-0.132035,0.024561,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,-13.815509,-13.815509,-13.815509,-13.815509
8,0,0.0,0,0.0,2,37.000000,-0.457683,0.100000,0.0,0.8,...,0.0,0.0,1.0,0.0,0.0,1.0,-13.815509,-13.815509,-13.815509,-13.815509


In [94]:
df_alr[['Administrative_alr','Informational_alr']].describe()

,Administrative_alr,Informational_alr
count,11610.000000,11610.000000
mean,-7.176819,-11.271649
std,5.915379,4.554694
min,-13.815509,-13.815509
25%,-13.815509,-13.815509
50%,-3.433987,-13.749732
75%,-1.945910,-13.227723
max,13.815509,13.815509


In [95]:
df_alr

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,Administrative_closed,Informational_closed,ProductRelated_closed,Administrative_Duration_closed,Informational_Duration_closed,ProductRelated_Duration_closed,Administrative_alr,Informational_alr,Administrative_Duration_alr,Informational_Duration_alr
1,0,0.0,0,0.0,2,64.000000,-0.457683,0.100000,0.000000,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,-13.815509,-13.815509,-13.815509,-13.815509
3,0,0.0,0,0.0,2,2.666667,0.573535,0.140000,0.000000,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,-13.815509,-13.815509,-13.815509,-13.815509
4,0,0.0,0,0.0,10,627.500000,-0.045196,0.050000,0.000000,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,-13.815509,-13.815509,-13.815509,-13.815509
5,0,0.0,0,0.0,19,154.216667,-0.132035,0.024561,0.000000,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,-13.815509,-13.815509,-13.815509,-13.815509
8,0,0.0,0,0.0,2,37.000000,-0.457683,0.100000,0.000000,0.8,...,0.0,0.0,1.0,0.0,0.0,1.0,-13.815509,-13.815509,-13.815509,-13.815509
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12325,3,145.0,0,0.0,53,1783.791667,-0.310366,0.029031,12.241717,0.0,...,0.053571,0.0,0.946429,0.075177,0.0,0.924823,-2.871680,-13.760450,-2.509763,-13.737357
12326,0,0.0,0,0.0,5,465.750000,-0.457683,0.021333,0.000000,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,-13.815509,-13.815509,-13.815509,-13.815509
12327,0,0.0,0,0.0,6,184.250000,1.261014,0.086667,0.000000,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,-13.815509,-13.815509,-13.815509,-13.815509
12328,4,75.0,0,0.0,15,346.000000,-0.457683,0.021053,0.000000,0.0,...,0.210526,0.0,0.789474,0.178147,0.0,0.821853,-1.321756,-13.579121,-1.528951,-13.619315


In [96]:
#for all columns in df_alr, if the datatype is bool, convert to int
bool_cols = df_alr.select_dtypes(include=["bool"]).columns
df_alr[bool_cols] = df_alr[bool_cols].astype(int)


In [97]:


# One-hot encode VisitorType 
df_alr = pd.get_dummies(df_alr, columns=['VisitorType'])

# One-hot encode Month
df_alr = pd.get_dummies(df_alr, columns=['Month'])

# One-hot encode Browser
df_alr = pd.get_dummies(df_alr, columns=['Browser'])

# One-hot encode Region
df_alr = pd.get_dummies(df_alr, columns=['Region'])

# One-hot encode TrafficType
df_alr = pd.get_dummies(df_alr, columns=['TrafficType'])

# One-hot encode OS
df_alr = pd.get_dummies(df_alr, columns=['OperatingSystems'])



df_alr



,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,TrafficType_19,TrafficType_20,OperatingSystems_1,OperatingSystems_2,OperatingSystems_3,OperatingSystems_4,OperatingSystems_5,OperatingSystems_6,OperatingSystems_7,OperatingSystems_8
1,0,0.0,0,0.0,2,64.000000,-0.457683,0.100000,0.000000,0.0,...,False,False,False,True,False,False,False,False,False,False
3,0,0.0,0,0.0,2,2.666667,0.573535,0.140000,0.000000,0.0,...,False,False,False,False,True,False,False,False,False,False
4,0,0.0,0,0.0,10,627.500000,-0.045196,0.050000,0.000000,0.0,...,False,False,False,False,True,False,False,False,False,False
5,0,0.0,0,0.0,19,154.216667,-0.132035,0.024561,0.000000,0.0,...,False,False,False,True,False,False,False,False,False,False
8,0,0.0,0,0.0,2,37.000000,-0.457683,0.100000,0.000000,0.8,...,False,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12325,3,145.0,0,0.0,53,1783.791667,-0.310366,0.029031,12.241717,0.0,...,False,False,False,False,False,True,False,False,False,False
12326,0,0.0,0,0.0,5,465.750000,-0.457683,0.021333,0.000000,0.0,...,False,False,False,False,True,False,False,False,False,False
12327,0,0.0,0,0.0,6,184.250000,1.261014,0.086667,0.000000,0.0,...,False,False,False,False,True,False,False,False,False,False
12328,4,75.0,0,0.0,15,346.000000,-0.457683,0.021053,0.000000,0.0,...,False,False,False,True,False,False,False,False,False,False


In [98]:


def drop_highest_freq_reference(df: pd.DataFrame, groups: list):
    """
    For each categorical group already OHE'd, drop the dummy column
    corresponding to the highest-frequency category (reference).
    
    Args:
        df (pd.DataFrame): Input dataframe with OHE columns
        groups (list): List of base names for categorical groups
    
    Returns:
        pd.DataFrame: Dataframe with reference columns dropped
        dict: Mapping of group -> dropped reference category
    """
    dropped_refs = {}

    for group in groups:
       
        group_cols = [c for c in df.columns if c.startswith(group + "_")]
        if not group_cols:
            continue

        
        ref_col = df[group_cols].sum().idxmax()
        dropped_refs[group] = ref_col

       
        df = df.drop(columns=[ref_col])

    return df, dropped_refs

# Example usage:
groups = ["Browser", "OperatingSystems", "Region", "VisitorType", "Month", "TrafficType"]
df_new, refs = drop_highest_freq_reference(df_alr, groups)
print("Reference categories dropped:", refs)


Reference categories dropped: {'Browser': 'Browser_2', 'OperatingSystems': 'OperatingSystems_2', 'Region': 'Region_1', 'VisitorType': 'VisitorType_Returning_Visitor', 'Month': 'Month_May', 'TrafficType': 'TrafficType_2'}


In [99]:
cols_to_drop = [
    "Administrative", "Informational", "ProductRelated",
    "Administrative_Duration", "Informational_Duration", "ProductRelated_Duration",
    "Administrative_closed", "Informational_closed", "ProductRelated_closed",
    "Administrative_Duration_closed", "Informational_Duration_closed", "ProductRelated_Duration_closed"
]
df_new = df_new.drop(columns=[col for col in cols_to_drop if col in df_new.columns])
df_new.shape

(11610, 67)

In [100]:
target = "Revenue"
cont_feats = ["Administrative_alr", "Informational_alr", "Administrative_Duration_alr", "Informational_Duration_alr",
              "PageValues", "BounceRates", "ExitRates"]
bin_feats  = ["Weekend", "SpecialDay"]
groups = ["Browser", "OperatingSystems", "Region", "VisitorType", "Month", "TrafficType"]
ohe_feats  = [c for c in df_new.columns if any(c.startswith(g+"_") for g in groups)]
X = df_new[cont_feats + bin_feats + ohe_feats].copy()
y = df_new[target].astype(int).values
X.shape



(11610, 66)

In [103]:
# Save transformed data for HMC/NUTS notebook
import pickle
import os
import numpy as np

# Create processed data directory if it doesn't exist
os.makedirs('../../data/processed', exist_ok=True)

# Ensure all data is numeric before saving (convert to float64 to avoid object dtype issues)
X_trs_array = X_trs.astype(float).values
X_vas_array = X_vas.astype(float).values
y_tr_array = y_tr.astype(int)
y_va_array = y_va.astype(int)

# Save training and validation sets
np.save('../../data/processed/X_trs.npy', X_trs_array)
np.save('../../data/processed/X_vas.npy', X_vas_array)
np.save('../../data/processed/y_tr.npy', y_tr_array)
np.save('../../data/processed/y_va.npy', y_va_array)

# Save scaler and feature information
with open('../../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../../data/processed/cont_feats.pkl', 'wb') as f:
    pickle.dump(cont_feats, f)

with open('../../data/processed/feature_names.pkl', 'wb') as f:
    pickle.dump(list(X_trs.columns), f)

print("Transformed data saved successfully!")
print(f"X_trs shape: {X_trs.shape}, saved to ../../data/processed/X_trs.npy")
print(f"X_vas shape: {X_vas.shape}, saved to ../../data/processed/X_vas.npy")
print(f"y_tr shape: {y_tr.shape}, saved to ../../data/processed/y_tr.npy")
print(f"y_va shape: {y_va.shape}, saved to ../../data/processed/y_va.npy")
print(f"Scaler saved to ../../data/processed/scaler.pkl")
print(f"Feature names saved to ../../data/processed/feature_names.pkl")


Transformed data saved successfully!
X_trs shape: (9288, 66), saved to ../../data/processed/X_trs.npy
X_vas shape: (2322, 66), saved to ../../data/processed/X_vas.npy
y_tr shape: (9288,), saved to ../../data/processed/y_tr.npy
y_va shape: (2322,), saved to ../../data/processed/y_va.npy
Scaler saved to ../../data/processed/scaler.pkl
Feature names saved to ../../data/processed/feature_names.pkl


In [102]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_trs, X_vas = X_tr.copy(), X_va.copy()
X_trs[cont_feats] = scaler.fit_transform(X_tr[cont_feats])
X_vas[cont_feats] = scaler.transform(X_va[cont_feats])


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, classification_report
clf = LogisticRegression(penalty="l2", C=1.0, solver="liblinear", max_iter=2000)
clf.fit(X_trs, y_tr)


p_tr = clf.predict_proba(X_trs)[:,1]
p_va = clf.predict_proba(X_vas)[:,1]
print("AUC (train, val):", roc_auc_score(y_tr,p_tr), roc_auc_score(y_va,p_va))
print("LogLoss (train, val):", log_loss(y_tr,p_tr), log_loss(y_va,p_va))
print(classification_report(y_va, (p_va>=0.5).astype(int)))


AUC (train, val): 0.8946180196370355 0.8862966704123345
LogLoss (train, val): 0.30407311841815854 0.30782517523218444
              precision    recall  f1-score   support

           0       0.89      0.98      0.93      1941
           1       0.75      0.38      0.50       381

    accuracy                           0.88      2322
   macro avg       0.82      0.68      0.72      2322
weighted avg       0.87      0.88      0.86      2322

